In [9]:
from glob import glob
import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

In [25]:
country = "extension_2"
regions_v = "4"

In [7]:
regions_datadir = "/data/uscuni-ulce/"
region_hulls = gpd.read_parquet(
    regions_datadir + "regions/" + "region_hulls_v3.parquet"
)  # delineation of existing regions 

Read data

In [11]:
new_folders = glob("/data/uscuni-ulce/extension2/*")
new_folders

['/data/uscuni-ulce/extension2/nor',
 '/data/uscuni-ulce/extension2/lie',
 '/data/uscuni-ulce/extension2/fin',
 '/data/uscuni-ulce/extension2/lit',
 '/data/uscuni-ulce/extension2/navarra',
 '/data/uscuni-ulce/extension2/dk',
 '/data/uscuni-ulce/extension2/bizkaia',
 '/data/uscuni-ulce/extension2/swe',
 '/data/uscuni-ulce/extension2/araba_alava',
 '/data/uscuni-ulce/extension2/swi',
 '/data/uscuni-ulce/extension2/gipuzkoa',
 '/data/uscuni-ulce/extension2/lux',
 '/data/uscuni-ulce/extension2/est']

In [12]:
# read the new data
gdf = pd.concat([
    gpd.read_parquet(f"{folder}/clean_3035.parquet") for folder in new_folders
], ignore_index=True)

In [13]:
gdf.shape

(29918481, 125)

## Split the buildings into regions

Assign building centroids to a grid, then cluster the grid points based on distances. This results in a split of the buildings into contiguous regions where all the 250m neighbours of a building are within the same region.

In [14]:
cents = gdf.centroid
gdf["x"], gdf["y"] = cents.x, cents.y
gdf["id"] = gdf.index.values
data = gdf[["x", "y", "id"]].copy()

In [15]:
data[["x_100", "y_100"]] = np.around(data[["x", "y"]], decimals=-2)
grid = data[["id", "x_100", "y_100"]].groupby(["x_100", "y_100"]).count().reset_index()

In [16]:
dbscan = DBSCAN(250, n_jobs=-1).fit(grid[["x_100", "y_100"]], sample_weight=grid["id"])
grid["labels"] = dbscan.labels_

Merge the results back into the dataframe. Then split the dataset into cores and non-cores. Cores are regions with more than 10_000 buildings.

In [17]:
# %%time
data = pd.merge(data, grid, "left", on=["x_100", "y_100"])

counts = data.labels.value_counts()

index = counts[counts > 10000].index
if -1 in index:
    index = index.drop(-1)

data["core"] = data.labels.isin(index)
grid["core"] = grid.labels.isin(index)


cores = data[data.core]
grid_cores = grid[grid.core]
grid_cores = gpd.GeoDataFrame(
    grid_cores["labels"],
    geometry=gpd.points_from_xy(grid_cores["x_100"], grid_cores["y_100"]),
    crs=3035,
)
grid_cores_dissolved = grid_cores.dissolve("labels")

grid_non_cores = grid[~grid.core]
grid_non_cores = gpd.GeoDataFrame(
    grid_non_cores["labels"],
    geometry=gpd.points_from_xy(grid_non_cores["x_100"], grid_non_cores["y_100"]),
    crs=3035,
)

grid_non_cores_clustered = grid_non_cores[grid_non_cores.labels != -1]
grid_non_cores_outliers = grid_non_cores[grid_non_cores.labels == -1]

grid_non_cores_clustered_dissolved = grid_non_cores_clustered.dissolve("labels")

Assign the non-core regions to the geographically nearest core region.

In [18]:
%%time
nearest = grid_cores.sindex.nearest(
    grid_non_cores_clustered_dissolved.geometry, return_all=False
)

CPU times: user 8.54 s, sys: 7.55 ms, total: 8.55 s
Wall time: 8.55 s


In [19]:
grid_non_cores_clustered_dissolved["nearest_core"] = grid_cores.labels.values[
    nearest[1]
]

In [20]:
nearest_outliers = grid_cores.sindex.nearest(
    grid_non_cores_outliers.geometry, return_all=False
)

In [21]:
grid_non_cores_outliers["nearest_core"] = grid_cores.labels.values[nearest_outliers[1]]

/home/martin/dev/urban_taxonomy/.pixi/envs/default/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [22]:
grid_non_cores = pd.concat(
    [
        grid_non_cores_clustered_dissolved.reset_index().explode(ignore_index=True),
        grid_non_cores_outliers,
    ],
    ignore_index=True,
)

Update the region labels based on the assignment

In [23]:
grid_non_cores["x_100"] = grid_non_cores.geometry.x
grid_non_cores["y_100"] = grid_non_cores.geometry.y

data = pd.merge(
    data,
    grid_non_cores[["x_100", "y_100", "nearest_core"]],
    "left",
    on=["x_100", "y_100"],
)

data["region"] = data.labels
data.loc[~data.core, "region"] = data.loc[~data.core, "nearest_core"]

data = data.rename(
    columns={
        "id_x": "id",
        "id_y": "weight",
        "labels": "dbscan_cluster",
    }
)

In [24]:
additional_region_hulls = (
    pd.concat(
        [
            grid_cores,
            grid_non_cores[["nearest_core", "geometry"]].rename(
                columns={"nearest_core": "labels"}
            ),
        ]
    )
    .dissolve("labels")
    .convex_hull.to_frame("convex_hull")
)

## Adjust the new region numbers and add to the existing regions

In [26]:
buffer = region_hulls.index.max() + 1000

In [27]:
additional_region_hulls.index = additional_region_hulls.index + buffer
data["region"] = data["region"] + buffer

In [28]:
additional_region_hulls.to_parquet(
    regions_datadir + "regions/" + f"{country}_regions_hull.parquet"
)

In [29]:
region_hulls = pd.concat((region_hulls, additional_region_hulls))

In [32]:
region_hulls.to_parquet(
    regions_datadir + "regions/" + f"region_hulls_v{regions_v}.parquet"
)

Show the largest regions.

In [33]:
data["region"].value_counts().sort_values().tail(20)

region
740437    226405
864429    232991
546972    241537
794352    250824
599278    259136
657430    265474
591101    275645
747994    289820
618519    298703
753898    301861
597798    311268
783776    320258
541712    329032
778471    335727
556455    392220
717071    415888
839116    430671
597702    440462
598030    454895
626843    811564
Name: count, dtype: int64

## Assign buildings to regions

In [35]:
region_mapping = data

In [36]:
%%time
for region_id, group in region_mapping.groupby("region"):
    region_id = int(region_id)

    buildings = gdf.iloc[group.id].copy()
    buildings["iid"] = buildings.index.values
    buildings.to_parquet(
        f"/data/uscuni-ulce/buildings_in_regions/buildings_{region_id}.pq"
    )

CPU times: user 1min 30s, sys: 12.8 s, total: 1min 43s
Wall time: 2min


## Check existing region ids

In [37]:
region_name = 657430
regions_datadir = "/data/uscuni-ulce/"
region_hulls = gpd.read_parquet(
    regions_datadir + "regions/" + f"region_hulls_v{regions_v}.parquet"
)
region_id, region_hull = (
    region_hulls.loc[region_name].name,
    region_hulls.loc[region_name].convex_hull,
)

In [38]:
gpd.GeoSeries([region_hull], name="geometry", crs="epsg:3035").explore()